In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ==============================
# SETTINGS
# ==============================
input_folder = "input_tiffs"      # folder containing .tif/.tiff files
output_folder = "output_results"  # folder to save outputs
BLOCK = 16  #Block size refers to dimensions in height and width which should be equal (e.g., block size 16 means each block is 16 X 16)

trim_left = 11
trim_top = 16
trim_right = 6
trim_bottom = 6

# ==============================
# TIFF IO BACKEND
# ==============================
try:
    import tifffile as tff
    def read_tiff(path): return tff.imread(path)
    def write_tiff(path, arr): tff.imwrite(path, arr)
    backend = "tifffile"
except Exception:
    import imageio.v2 as iio
    def read_tiff(path): return iio.imread(path)
    def write_tiff(path, arr): iio.imwrite(path, arr, format='TIFF')
    backend = "imageio"

print("Using backend:", backend)

# ==============================
# PREP OUTPUT FOLDER
# ==============================
os.makedirs(output_folder, exist_ok=True)

# ==============================
# FIND ALL TIFF FILES
# ==============================
tiff_files = [
    f for f in os.listdir(input_folder)
    if f.lower().endswith((".tif", ".tiff"))
]

if len(tiff_files) == 0:
    raise RuntimeError("No TIFF files found in input folder.")

print("Found", len(tiff_files), "TIFF files.")

# ==============================
# PROCESS EACH FILE
# ==============================
for filename in tiff_files:

    print("\nProcessing:", filename)
    path = os.path.join(input_folder, filename)
    arr = read_tiff(path)
    arr = np.array(arr)

    # ------------------------------
    # Normalize to (frames, H, W)
    # ------------------------------
    if arr.ndim == 2:
        stack = arr[np.newaxis, ...]
    elif arr.ndim == 3:
        if arr.shape[2] in (3,4):  # RGB
            rgb = arr[..., :3].astype(np.float32)
            gray = (0.299*rgb[...,0] +
                    0.587*rgb[...,1] +
                    0.114*rgb[...,2])
            stack = gray[np.newaxis, ...]
        else:
            stack = arr
    elif arr.ndim == 4:
        frames = []
        for k in range(arr.shape[0]):
            imk = arr[k]
            if imk.ndim == 3 and imk.shape[2] in (3,4):
                rgb = imk[..., :3].astype(np.float32)
                gray = (0.299*rgb[...,0] +
                        0.587*rgb[...,1] +
                        0.114*rgb[...,2])
                frames.append(gray)
            else:
                frames.append(imk.squeeze())
        stack = np.stack(frames, axis=0)
    else:
        raise ValueError("Unsupported TIFF format.")

    stack = np.array(stack)
    n_frames, H, W = stack.shape
    print("  Frames:", n_frames, " Size:", H, "x", W)

    # ------------------------------
    # Trim and crop size calculation
    # ------------------------------
    H_after_trim = H - trim_top - trim_bottom
    W_after_trim = W - trim_left - trim_right

    if H_after_trim <= 0 or W_after_trim <= 0:
        print("  Skipping (trim removes entire image)")
        continue

    Ht_ds = (H_after_trim // BLOCK) * BLOCK
    Wt_ds = (W_after_trim // BLOCK) * BLOCK

    if Ht_ds == 0 or Wt_ds == 0:
        print("  Skipping (image too small after trim)")
        continue

    nh = Ht_ds // BLOCK
    nw = Wt_ds // BLOCK

    down_frames = np.zeros((n_frames, nh, nw), dtype=stack.dtype)

    # ------------------------------
    # Process frames
    # ------------------------------
    for f in range(n_frames):

        frame = stack[f]
        frame_trim = frame[
            trim_top:H-trim_bottom,
            trim_left:W-trim_right
        ]
        frame_crop = frame_trim[:Ht_ds, :Wt_ds]
        blocks = frame_crop.reshape(nh, BLOCK, nw, BLOCK)

        block_means = blocks.mean(axis=(1,3))
        block_ranges = blocks.max(axis=(1,3)) - blocks.min(axis=(1,3))

        # Save block metrics CSV
        csv_rows = []
        for i in range(nh):
            for j in range(nw):
                csv_rows.append([
                    i, j,
                    float(block_means[i,j]),
                    float(block_ranges[i,j])
                ])

        base = os.path.splitext(filename)[0]
        csv_name = f"{base}_frame{f:03d}_metrics.csv"
        csv_path = os.path.join(output_folder, csv_name)

        np.savetxt(
            csv_path,
            np.array(csv_rows),
            delimiter=",",
            header="block_row,block_col,mean,range",
            comments=""
        )

        # Compute mode per block
        rep = np.zeros((nh, nw), dtype=frame_crop.dtype)
        for i in range(nh):
            for j in range(nw):
                b = blocks[i, :, j, :].ravel()

                if frame_crop.dtype == np.uint8:
                    counts = np.bincount(b, minlength=256)
                    rep[i,j] = np.argmax(counts)
                else:
                    vals, cnts = np.unique(b, return_counts=True)
                    rep[i,j] = vals[np.argmax(cnts)]

        down_frames[f] = rep

    # ------------------------------
    # Save downsampled TIFF
    # ------------------------------
    out_tiff_name = os.path.splitext(filename)[0] + "_downsampled.tif"
    out_tiff_path = os.path.join(output_folder, out_tiff_name)

    write_tiff(out_tiff_path, down_frames)
    print("  Saved:", out_tiff_name)

print("\nProcessing complete.")

Using backend: tifffile
Found 1 TIFF files.

Processing: Mitochondria__even_eroded_rm50pct.tif
  Frames: 24  Size: 1608 x 1365
  Saved: Mitochondria__even_eroded_rm50pct_downsampled.tif

Processing complete.
